# 03 — Stage 1 Instrument Embedding (Colab + Drive)

MIL + attention → 64-d song embedding. **GPU On**.

Needs: 00 + 01. Writes `features/instrument/` on Drive.


## Colab + Drive (every notebook)

1. Open in **Google Colab**.
2. Run **Mount Drive** and click **Allow**.
3. Shared folder: `/content/drive/MyDrive/MTG_Instrument`
4. GPU **On** only for 02, 03, 07. Off for 00, 01, 04–06, 09.
5. Do **not** re-download mels after notebook 00.


In [ ]:
!pip install -q scikit-learn tqdm


## Mount Drive


In [ ]:
from pathlib import Path
import os

DRIVE_ROOT = Path("/content/drive/MyDrive/MTG_Instrument")

if not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

for sub in ["dataset/logmel_songs", "annotations", "features", "checkpoints", "results"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["MTG_ROOT"] = str(DRIVE_ROOT)
print("Drive ready:", DRIVE_ROOT)


In [ ]:
from pathlib import Path
import os, json, random, re, shutil, socket, time, urllib.request
import numpy as np
import pandas as pd

DRIVE_ROOT = Path(os.environ.get("MTG_ROOT", "/content/drive/MyDrive/MTG_Instrument"))
ROOT = DRIVE_ROOT
MEL_DIR = ROOT / "dataset" / "logmel_songs"
MEL_CACHE = Path("/content/mel_cache")
MEL_CACHE.mkdir(parents=True, exist_ok=True)
ANN_DIR = ROOT / "annotations"
FEAT_DIR = ROOT / "features"
CKPT_DIR = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
MANIFEST = ROOT / "dataset" / "song_manifest.csv"
RAW_ANN = "https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master/data"
NEEDED_ANN = [
    "splits/split-0/autotagging_genre-train.tsv",
    "splits/split-0/autotagging_genre-validation.tsv",
    "splits/split-0/autotagging_genre-test.tsv",
    "splits/split-0/autotagging_instrument-train.tsv",
    "splits/split-0/autotagging_instrument-validation.tsv",
    "splits/split-0/autotagging_instrument-test.tsv",
    "autotagging_genre.tsv",
    "autotagging_instrument.tsv",
]
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def check_internet(host="github.com", port=443, timeout=5) -> bool:
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False


def normalize_track_id(raw) -> str | None:
    m = re.search(r"(\d+)", str(raw))
    return f"{int(m.group(1)):07d}" if m else None


def ensure_annotations():
    dest_train = ANN_DIR / "splits" / "split-0" / "autotagging_genre-train.tsv"
    if dest_train.exists():
        return
    if not check_internet():
        raise FileNotFoundError("Split TSVs missing and no Internet. Enable Internet and re-run.")
    print("Downloading annotation TSVs to Drive...")
    for rel in NEEDED_ANN:
        dest = ANN_DIR / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"{RAW_ANN}/{rel}", dest)
        print(" ", dest)


def load_split_ids(split: str, subset: str = "genre") -> set[str]:
    """First column only — extra tag tabs break pandas read_csv."""
    name = f"autotagging_{subset}-{split}.tsv"
    path = ANN_DIR / "splits" / "split-0" / name
    if not path.exists():
        path = ANN_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    ids = set()
    with open(path, encoding="utf-8", errors="replace") as f:
        f.readline()
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            tid = normalize_track_id(line.split("\t")[0])
            if tid:
                ids.add(tid)
    print(f"{split:12s} {len(ids):6d} ids ← {path}")
    return ids


def iter_tsv_rows(path: Path):
    """Yield dict with TRACK_ID and remaining fields joined as TAGS."""
    with open(path, encoding="utf-8", errors="replace") as f:
        header = f.readline().strip().split("\t")
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if not parts:
                continue
            row = {"TRACK_ID": parts[0]}
            if len(parts) >= 6:
                row["TAGS"] = "\t".join(parts[5:])
            elif len(parts) > 1:
                row["TAGS"] = parts[-1]
            else:
                row["TAGS"] = ""
            yield row


def load_mel_npy(mel_abs, retries=5, pause=2.0):
    """Load mel from Drive with retries; cache on Colab disk to avoid FUSE drops."""
    mel_abs = Path(mel_abs)
    sid = normalize_track_id(mel_abs.stem) or mel_abs.stem.replace("/", "_")
    cached = MEL_CACHE / f"{sid}.npy"
    if cached.exists():
        try:
            return np.load(cached)
        except (OSError, ValueError):
            cached.unlink(missing_ok=True)

    last_err = None
    for attempt in range(retries):
        try:
            arr = np.load(mel_abs, mmap_mode=None)
            arr = np.asarray(arr, dtype=np.float32)
            np.save(cached, arr)
            return arr
        except (OSError, ValueError) as e:
            last_err = e
            if attempt + 1 < retries:
                time.sleep(pause * (attempt + 1))
    nbytes = mel_abs.stat().st_size if mel_abs.exists() else "missing"
    raise RuntimeError(
        f"Bad/truncated mel — re-download its shard in notebook 00: {mel_abs} "
        f"({nbytes} bytes on Drive). {last_err}"
    ) from last_err


def scan_bad_mels(df, label="manifest"):
    from tqdm.auto import tqdm

    bad = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"scan {label}"):
        try:
            load_mel_npy(row["mel_abs"])
        except Exception as e:
            bad.append({"song_id": str(row["song_id"]), "mel_abs": row["mel_abs"], "error": str(e)})
    if bad:
        out = RESULTS_DIR / f"bad_mels_{label}.json"
        out.write_text(json.dumps(bad, indent=2))
        print(f"WARNING: {len(bad)} bad mels → {out}")
    else:
        print(f"scan {label}: all {len(df)} mels OK (cache: {MEL_CACHE})")
    return bad


ensure_annotations()
print("ROOT   ", ROOT)
print("MEL_DIR", MEL_DIR, "npy=", len(list(MEL_DIR.rglob("*.npy"))))
print("ANN_DIR", ANN_DIR)
print("MANIFEST", MANIFEST, "exists=", MANIFEST.exists())


## Train + export embeddings


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not MANIFEST.exists():
    raise FileNotFoundError("Run 01 first")
manifest = pd.read_csv(MANIFEST)
manifest["song_id"] = manifest["song_id"].astype(str).map(lambda s: normalize_track_id(s) or s)
EMBED_DIM, MAX_WINDOWS = 64, 12
N_MELS, N_FRAMES = 96, 1366
song_ids = manifest["song_id"].astype(str).tolist()
id_to_idx = {s: i for i, s in enumerate(song_ids)}

tag_to_idx, rows = {}, {s: set() for s in song_ids}
for path in [ANN_DIR/"autotagging_instrument.tsv", *ANN_DIR.rglob("*instrument*.tsv")]:
    if not Path(path).exists():
        continue
    for rec in iter_tsv_rows(Path(path)):
        sid = normalize_track_id(rec["TRACK_ID"])
        if sid not in rows:
            continue
        for tag in rec.get("TAGS", "").replace("|", "\t").split("\t"):
            leaf = tag.strip().split("/")[-1].split("---")[-1]
            if leaf and leaf.lower() not in {"nan", "tags", ""}:
                tag_to_idx.setdefault(leaf, len(tag_to_idx))
                rows[sid].add(leaf)
    if tag_to_idx:
        print("instruments", path, len(tag_to_idx)); break
INST_NAMES = [None]*len(tag_to_idx)
for t,i in tag_to_idx.items(): INST_NAMES[i]=t
Y = np.zeros((len(song_ids), len(INST_NAMES)), np.float32)
for sid, tags in rows.items():
    i = id_to_idx[sid]
    for t in tags: Y[i, tag_to_idx[t]] = 1.0

class WindowMIL(Dataset):
    def __init__(self, df, max_windows=MAX_WINDOWS, n_mels=N_MELS, n_frames=N_FRAMES):
        self.df = df.reset_index(drop=True)
        self.max_windows, self.n_mels, self.n_frames = max_windows, n_mels, n_frames

    def __len__(self):
        return len(self.df)

    def _fix2d(self, x):
        """Force every window to exactly (n_mels, n_frames)."""
        x = np.asarray(x, dtype=np.float32)
        while x.ndim > 2:
            x = np.squeeze(x, axis=0)
        if x.ndim != 2:
            raise ValueError(f"expected 2D mel window, got {x.shape}")
        if x.shape[0] != self.n_mels and x.shape[1] == self.n_mels:
            x = x.T
        if x.shape[0] > self.n_mels:
            x = x[: self.n_mels]
        elif x.shape[0] < self.n_mels:
            x = np.pad(x, ((0, self.n_mels - x.shape[0]), (0, 0)))
        if x.shape[1] > self.n_frames:
            x = x[:, : self.n_frames]
        elif x.shape[1] < self.n_frames:
            x = np.pad(x, ((0, 0), (0, self.n_frames - x.shape[1])))
        if x.shape != (self.n_mels, self.n_frames):
            raise RuntimeError(f"mel fix failed: {x.shape}")
        return x

    def __getitem__(self, i):
        row = self.df.iloc[i]
        raw = load_mel_npy(row["mel_abs"])
        if raw.ndim == 2:
            raw = raw[None, ...]
        W = raw.shape[0]
        if W >= self.max_windows:
            windows, mask = raw[: self.max_windows], np.ones(self.max_windows, np.float32)
        else:
            windows = np.concatenate(
                [raw, np.zeros((self.max_windows - W, *raw.shape[1:]), raw.dtype)]
            )
            mask = np.array([1] * W + [0] * (self.max_windows - W), np.float32)
        out = np.zeros((self.max_windows, self.n_mels, self.n_frames), np.float32)
        for j, w in enumerate(windows):
            out[j] = self._fix2d(w)
        y = Y[id_to_idx[str(row["song_id"])]]
        return (
            torch.from_numpy(out[:, None]),
            torch.from_numpy(mask),
            torch.from_numpy(y),
            str(row["song_id"]),
        )

def make_loader(split, bs=8, shuffle=False):
    sub = manifest[manifest.split==split]
    assert set(sub.split.unique())=={split}
    return DataLoader(WindowMIL(sub), batch_size=bs, shuffle=shuffle, num_workers=0)

class Stage1(nn.Module):
    def __init__(self, n_tags, emb=EMBED_DIM):
        super().__init__()
        self.cnn = nn.Sequential(nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                 nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1)))
        self.proj = nn.Linear(64, emb)
        self.score = nn.Linear(emb, 1)
        self.head = nn.Linear(emb, n_tags)
    def forward(self, x, mask):
        B,W,C,M,T = x.shape
        H = self.proj(self.cnn(x.reshape(B*W,C,M,T)).flatten(1)).reshape(B,W,-1)
        logits = self.score(H).squeeze(-1).masked_fill(mask<0.5, -1e9)
        w = torch.softmax(logits, -1)
        z = (H * w.unsqueeze(-1)).sum(1)
        return self.head(z), z, w

model = Stage1(Y.shape[1]).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss()

def macro_map(yt, yp):
    s=[]
    for k in range(yt.shape[1]):
        if yt[:,k].sum() in (0,len(yt)): continue
        try: s.append(average_precision_score(yt[:,k], yp[:,k]))
        except ValueError: pass
    return float(np.mean(s)) if s else float("nan")

@torch.no_grad()
def eval_split(dl):
    model.eval(); ys,ps=[],[]
    for x,mask,y,_ in dl:
        logits,_,_ = model(x.to(DEVICE), mask.to(DEVICE))
        ps.append(torch.sigmoid(logits).cpu().numpy()); ys.append(y.numpy())
    return macro_map(np.concatenate(ys), np.concatenate(ps))

tr, va, te = make_loader("train", shuffle=True), make_loader("validation"), make_loader("test")
_x, _m, _y, _ = next(iter(tr))
print("preflight batch", tuple(_x.shape), "expect (bs, 12, 1, 96, 1366)")
assert _x.shape[2:] == (1, N_MELS, N_FRAMES), f"re-run this entire cell — got {_x.shape}"
SCAN_MELS = False
if SCAN_MELS:
    bad = scan_bad_mels(manifest, "all")
    if bad:
        raise RuntimeError(f"{len(bad)} bad mels — see {RESULTS_DIR}/bad_mels_all.json")
best_macro_map = 0.0
ckpt = CKPT_DIR/"stage1"; ckpt.mkdir(parents=True, exist_ok=True)
for epoch in range(1, 9):
    model.train(); total=0
    for x,mask,y,_ in tqdm(tr, leave=False):
        x,mask,y = x.to(DEVICE), mask.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); logits,_,_=model(x,mask); loss=crit(logits,y); loss.backward(); opt.step()
        total += loss.item()*len(x)
    vm = eval_split(va)
    print(epoch, "val_map", vm)
    if vm > best_macro_map:
        best_macro_map = vm
        torch.save({"model": model.state_dict(), "best_macro_map": best_macro_map, "tags": INST_NAMES}, ckpt/"best.pt")
        print("  ✓", best_macro_map)

state = torch.load(ckpt/"best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model"]); model.eval()
embeds, ids = [], []
with torch.no_grad():
    for x,mask,y,sid in tqdm(DataLoader(WindowMIL(manifest), batch_size=8, num_workers=0)):
        _, z, _ = model(x.to(DEVICE), mask.to(DEVICE))
        embeds.append(z.cpu().numpy()); ids.extend(list(sid))
E = np.concatenate(embeds, 0)
out = FEAT_DIR/"instrument"; out.mkdir(parents=True, exist_ok=True)
np.save(out/"instrument_embeddings.npy", E)
(out/"song_ids.json").write_text(json.dumps(ids))
print("saved", E.shape, "test", eval_split(te))
